# The Volatility Complex: Topological Structure of Sector Stock Volatility

This notebook constructs **Vietoris-Rips complexes** from the rolling volatility of S&P 500 sector stocks, then studies how the complex evolves over time and computes its **persistent homology**.

**Pipeline:**
1. Download sector stock prices (e.g., Information Technology)
2. Compute log-returns → rolling annualized volatility
3. Correlation of volatilities → distance matrix $D_{ij} = \sqrt{2(1 - \rho_{ij})}$
4. Build Vietoris-Rips complex: edge $(i,j)$ included when $D_{ij} < r$
5. Study rolling complexes over time (f-vector evolution)
6. Persistent homology via filtration over radius $r$

**Key idea:** When volatilities are highly correlated (market stress), the complex becomes dense — all stocks move together. In calm periods, the complex fragments into sector clusters.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from volatility_complex import (
    load_sp500_sector,
    load_sector_etfs,
    compute_log_returns,
    rolling_correlation,
    rolling_distance,
    from_distance_matrix,
    from_correlation,
    rolling_complexes,
    f_vector,
    plot_graph,
    plot_complex_summary,
    plot_adjacency_heatmap,
)

## 1. Download Sector Stock Data

We download ~50 stocks from the **Information Technology** sector of the S&P 500. The date range 2007–2024 covers the 2008 financial crisis, the 2020 COVID crash, and the 2022 tech drawdown — all events that should produce visible topological signatures.

In [ ]:
start = "2007-01-01"
end = "2024-06-30"

tech_prices = load_sp500_sector("tech", start, end, n=50, seed=42)
print(f"Shape: {tech_prices.shape}")
print(f"Tickers: {list(tech_prices.columns)}")

## 2. Compute Log-Returns and Volatility

We compute daily log-returns, then estimate **rolling annualized volatility** as $\sigma_t = \text{std}(r_{t-w:t}) \times \sqrt{252}$ with a 30-day window. The volatility time series (not raw returns) is the input to our complex — we're studying how *risk profiles* co-move, not price levels.

In [ ]:
tech_log_returns = compute_log_returns(tech_prices)
tech_cum_returns = np.exp(np.cumsum(tech_log_returns))
tech_volatility = tech_log_returns.rolling(window=30).std() * np.sqrt(252)

print(f"Volatility shape: {tech_volatility.dropna().shape}")

# Plot a few volatility series
fig, ax = plt.subplots(figsize=(14, 5))
tech_volatility.iloc[:, :5].plot(ax=ax, alpha=0.8)
ax.set_title("Rolling 30-day Annualized Volatility (first 5 stocks)")
ax.set_ylabel("Volatility")
plt.show()

## 3. Correlation → Distance Matrix

From the volatility time series, we compute the full-sample **correlation matrix** $\rho_{ij}$ and convert it to a **distance metric**:

$$D_{ij} = \sqrt{2(1 - \rho_{ij})}$$

This is a proper metric on $[0, 2]$: $D=0$ means perfectly correlated volatilities, $D=\sqrt{2} \approx 1.414$ means uncorrelated, and $D=2$ means perfectly anti-correlated.

In [ ]:
tech_corr = tech_volatility.corr()
tech_dist = np.sqrt(2 * (1 - tech_corr))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(tech_corr, cmap='RdBu_r', center=0, xticklabels=False, yticklabels=False, ax=axes[0])
axes[0].set_title('Volatility Correlation Matrix')
sns.heatmap(tech_dist, cmap='magma_r', xticklabels=False, yticklabels=False, ax=axes[1])
axes[1].set_title('Distance Matrix D = √(2(1-ρ))')
plt.tight_layout()
plt.show()

## 4. Build a Static Vietoris-Rips Complex

The **Vietoris-Rips complex** at radius $r$ includes:
- A vertex for each stock
- An edge $(i,j)$ whenever $D_{ij} < r$
- A triangle $(i,j,k)$ whenever all three pairwise distances are $< r$ (flag complex)
- Higher simplices by the same clique rule

**Choosing $r$:** Since $D = \sqrt{2(1-\rho)}$:
- $r \approx 0.5 \Rightarrow \rho > 0.875$ (very strong correlation only) → sparse
- $r = 1.0 \Rightarrow \rho > 0.5$ (moderate+) → moderate density
- $r \approx 1.414 \Rightarrow \rho > 0$ (any positive correlation) → dense

In [ ]:
# Build Rips complex at radius r=0.9 from the full-sample distance matrix
sc = from_distance_matrix(tech_dist, radius=0.9, max_dim=3)
print(f"f-vector: {f_vector(sc)}")
print(f"  {sc.n_vertices} vertices, {sc.n_edges} edges, {sc.n_triangles} triangles")

# Sweep radius to see how complex density changes
radii = np.arange(0.3, 1.5, 0.05)
edge_counts = []
tri_counts = []
for r in radii:
    sc_r = from_distance_matrix(tech_dist, radius=r, max_dim=2)
    edge_counts.append(sc_r.n_edges)
    tri_counts.append(sc_r.n_triangles)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(radii, edge_counts, label='Edges')
ax.plot(radii, tri_counts, label='Triangles')
ax.set_xlabel('Radius r')
ax.set_ylabel('Count')
ax.set_title('Simplex Counts vs. Rips Radius (full-sample distance matrix)')
ax.legend()
plt.show()

## 5. Visualize the 1-Skeleton

The 1-skeleton (vertices + edges) of the Rips complex is a graph where connected stocks have similar volatility profiles. The Kamada-Kawai layout places tightly-connected clusters close together.

In [ ]:
sc = from_distance_matrix(tech_dist, radius=0.9, max_dim=1)

fig, ax = plt.subplots(figsize=(12, 10))
plot_graph(sc, ax=ax, layout='kamada_kawai', title='Tech Volatility Complex (r=0.9, full sample)')
plt.show()

## 6. Rolling Distance Matrices

Instead of a single distance matrix from the entire sample, we now compute **rolling** distance matrices using a 60-day window. This gives us a distance matrix at each trading day, capturing how the volatility correlation structure evolves.

In [ ]:
import ipywidgets as widgets

dist_matrices = rolling_distance(tech_volatility, window=60)
timestamps = list(dist_matrices.keys())
print(f"{len(timestamps)} rolling distance matrices computed")

def plot_heatmap(time_window):
    sns.heatmap(dist_matrices[timestamps[time_window]], cmap='magma_r', xticklabels=False, yticklabels=False)
    plt.title(f"Rolling Distance Matrix — {timestamps[time_window].date().strftime('%B %d, %Y')}")

widgets.interact(plot_heatmap, time_window=widgets.IntSlider(min=0, max=len(timestamps)-1, step=1, value=0))

## 7. Rolling Rips Complexes — Evolution Over Time

We build a Rips complex at **each** time step (fixed radius, varying distance matrix). The f-vector at each date tells us how interconnected the market's volatility structure is:
- **Many edges** → volatilities are correlated (stress / contagion)
- **Few edges** → idiosyncratic volatility regimes (calm / differentiated)

In [ ]:
# Build rolling Rips complexes (max_dim=3 to see higher-order structure)
rolling_sc = rolling_complexes(
    tech_volatility,
    window=60,
    method="distance",
    threshold=0.9,
    max_dim=3,
)

dates = list(rolling_sc.keys())
print(f"{len(dates)} complexes built")
print(f"Example f-vector at {dates[100].date()}: {f_vector(rolling_sc[dates[100]])}")

In [ ]:
# Plot simplex counts over time
x = list(range(1, len(rolling_sc)))
y_edges = [f_vector(rolling_sc[dates[i]])[1] for i in x]
y_tris = [f_vector(rolling_sc[dates[i]])[2] for i in x]
y_tets = [f_vector(rolling_sc[dates[i]])[3] for i in x]

fig, ax = plt.subplots(figsize=(24, 8))
ax.plot(x, y_edges, label='Edges (1-simplices)', alpha=0.9)
ax.plot(x, y_tris, label='Triangles (2-simplices)', alpha=0.9)
ax.plot(x, y_tets, label='Tetrahedra (3-simplices)', alpha=0.9)

# Date labels on x-axis
step = len(x) // 25
tick_pos = x[::step]
tick_labels = [str(dates[i].date()) for i in tick_pos]
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, rotation=45)

ax.set_xlabel('Time')
ax.set_ylabel('Number of Simplices')
ax.set_title('Tech Volatility Complex — Simplex Counts Over Time (r=0.9)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Interactive Rolling Rips Explorer

Adjust the radius, max dimension, and window size, then scrub through time to see how the complex topology changes. The optional **return × degree coloring** shades nodes red (up) or blue (down) by their return direction, with intensity proportional to vertex degree.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

def interactive_rolling_rips(returns):
    """Interactive widget for exploring rolling Vietoris-Rips complexes."""
    radius_slider  = widgets.FloatSlider(value=1.0, min=0.5, max=1.414, step=0.01, description='radius r')
    max_dim_slider = widgets.IntSlider(value=1, min=1, max=3, step=1, description='max_dim')
    window_slider  = widgets.IntSlider(value=60, min=20, max=252, step=10, description='window')
    time_slider    = widgets.IntSlider(value=0, min=0, max=0, step=1, description='time_window')
    color_toggle   = widgets.Checkbox(value=False, description='return × degree color')

    recompute_btn = widgets.Button(description='Build complexes')
    out = widgets.Output()
    state = {'rolling_sc': {}, 'timestamps': []}
    rd_cmap = plt.cm.RdBu_r

    def _return_degree_colors(sc, timestamp):
        deg = np.zeros(sc.n_vertices)
        for i, j in sc.edges:
            deg[i] += 1; deg[j] += 1
        max_deg = deg.max()
        deg_norm = deg / max_deg if max_deg > 0 else deg
        window = window_slider.value
        idx = returns.index.get_loc(timestamp)
        start_idx = max(0, idx - window + 1)
        win_ret = returns.iloc[idx] / returns.iloc[start_idx] - 1
        colors = []
        for vi, ticker in enumerate(sc.vertices):
            ret = win_ret.get(ticker, 0)
            sign = 1.0 if ret >= 0 else -1.0
            colors.append(rd_cmap(0.5 + 0.5 * sign * deg_norm[vi]))
        return colors

    def _recompute(_=None):
        with out:
            clear_output(wait=True)
            print('Computing rolling complexes...')
            state['rolling_sc'] = rolling_complexes(
                returns, window=window_slider.value, method='distance',
                threshold=radius_slider.value, max_dim=max_dim_slider.value)
            state['timestamps'] = list(state['rolling_sc'].keys())
            time_slider.max = len(state['timestamps']) - 1
        _plot_current()

    def _plot_current(change=None):
        ts = state['timestamps']
        if not ts: return
        idx = time_slider.value
        sc = state['rolling_sc'][ts[idx]]
        node_color = _return_degree_colors(sc, ts[idx]) if color_toggle.value else None
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 8))
            plot_graph(sc, ax=ax, layout='kamada_kawai', node_color=node_color,
                       title=f"Rips Complex — {ts[idx].date().strftime('%B %d, %Y')}  (r={radius_slider.value:.2f})")
            if color_toggle.value:
                sm = ScalarMappable(cmap=rd_cmap, norm=Normalize(vmin=-1, vmax=1))
                sm.set_array([])
                cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
                cbar.set_label('← Declining       Increasing →', fontsize=9)
            plt.show()

    recompute_btn.on_click(_recompute)
    time_slider.observe(_plot_current, names='value')
    color_toggle.observe(_plot_current, names='value')
    display(widgets.HBox([radius_slider, max_dim_slider, window_slider, recompute_btn]))
    display(widgets.HBox([time_slider, color_toggle]), out)
    _recompute()

interactive_rolling_rips(tech_volatility.iloc[:, :50])

## 9. Animate the Rolling Rips Complex

Generate a GIF showing how the volatility complex evolves over time. Nodes are colored by return direction (red = up, blue = down) with intensity proportional to vertex degree (more connected = more saturated).

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from IPython.display import Image

def animate_rolling_rips(
    returns, window=60, radius=1.0, max_dim=1, interval=80,
    save_path='rips_complex.gif', figsize=(10, 8), dpi=100, step=1,
    alpha=0.3, return_degree_color=False,
):
    """Animate rolling Vietoris-Rips complexes as a GIF."""
    print('Computing rolling complexes...')
    rc = rolling_complexes(returns, window=window, method='distance',
                           threshold=radius, max_dim=max_dim)
    ts = list(rc.keys())[::step]
    print(f'{len(ts)} frames to render')

    fig, ax = plt.subplots(figsize=figsize)
    pos = None
    rd_cmap = plt.cm.RdBu_r

    if return_degree_color:
        sm = ScalarMappable(cmap=rd_cmap, norm=Normalize(vmin=-1, vmax=1))
        sm.set_array([])
        fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02).set_label(
            '← Declining       Increasing →', fontsize=9)

    def _rd_colors(sc, frame_idx):
        deg = np.zeros(sc.n_vertices)
        for i, j in sc.edges:
            deg[i] += 1; deg[j] += 1
        max_deg = deg.max()
        deg_norm = deg / max_deg if max_deg > 0 else deg
        t = ts[frame_idx]
        idx = returns.index.get_loc(t)
        s = max(0, idx - window + 1)
        win_ret = returns.iloc[idx] / returns.iloc[s] - 1
        return [rd_cmap(0.5 + 0.5 * (1.0 if win_ret.get(tk, 0) >= 0 else -1.0) * deg_norm[vi])
                for vi, tk in enumerate(sc.vertices)]

    def update(frame):
        nonlocal pos
        ax.clear()
        sc = rc[ts[frame]]
        nc = _rd_colors(sc, frame) if return_degree_color else None
        _, new_pos = plot_graph(sc, ax=ax, pos=pos, layout='kamada_kawai', node_color=nc,
                                title=f"{ts[frame].date():%B %d, %Y}  (r={radius:.2f})")
        if pos is not None:
            pos = {n: alpha * np.array(new_pos[n]) + (1-alpha) * np.array(pos[n])
                   if n in pos else new_pos[n] for n in new_pos}
        else:
            pos = new_pos

    anim = FuncAnimation(fig, update, frames=len(ts), interval=interval)
    anim.save(save_path, writer=PillowWriter(fps=max(1, 1000 // interval)), dpi=dpi)
    plt.close(fig)
    print(f'Saved to {save_path}')
    return Image(filename=save_path)

animate_rolling_rips(tech_volatility.iloc[:, :50], radius=0.9, alpha=0.12,
                     step=1, dpi=80, interval=80, return_degree_color=True)

## 10. Persistent Homology

For a **fixed time window**, we vary the Rips radius $r$ from 0 to 2 to get a **filtration** — a nested sequence of complexes. Persistent homology tracks when topological features (connected components, loops, voids) are **born** and **die** as $r$ increases.

- **$H_0$ (connected components):** Start with $n$ isolated vertices. As $r$ grows, edges appear and components merge. Long bars = well-separated clusters of stocks.
- **$H_1$ (loops):** A loop is born when a cycle of edges forms without being filled by a triangle. Long $H_1$ bars = robust cyclic correlation structures.
- **$H_2$ (voids):** Enclosed cavities bounded by triangles.

This uses [GUDHI](https://gudhi.inria.fr/) for the Rips filtration and persistence computation.

In [ ]:
import gudhi

# Pick a specific time window
dist_matrices = rolling_distance(tech_volatility, window=60)
dates = list(dist_matrices.keys())
idx = 500
date = dates[idx]

D = dist_matrices[date].values
print(f"Computing persistent homology for window ending {str(date)[:10]}")

# Build Rips complex with gudhi
rips = gudhi.RipsComplex(distance_matrix=D, max_edge_length=2.0)
simplex_tree = rips.create_simplex_tree(max_dimension=3)
persistence = simplex_tree.persistence()

# Plot barcode and diagram
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
gudhi.plot_persistence_barcode(persistence, axes=axes[0])
axes[0].set_title(f'Persistence Barcode — {str(date)[:10]}')
gudhi.plot_persistence_diagram(persistence, axes=axes[1])
axes[1].set_title(f'Persistence Diagram — {str(date)[:10]}')
plt.tight_layout()
plt.show()

## 11. Interactive Persistence Explorer

Scrub through time to see how the persistence barcode changes. This precomputes persistence for every time window (may take a few minutes), then lets you interactively browse.

In [ ]:
import ipywidgets as widgets

# Precompute persistence for all windows
dist_matrices = rolling_distance(tech_volatility, window=60)
dates = list(dist_matrices.keys())

persistence_cache = {}
for i in range(0, len(dates)):
    D = dist_matrices[dates[i]].values
    rips = gudhi.RipsComplex(distance_matrix=D, max_edge_length=2.0)
    st = rips.create_simplex_tree(max_dimension=3)
    persistence_cache[i] = st.persistence()

print(f'Precomputed {len(persistence_cache)} persistence diagrams')

def persistence_widget(date_idx=0):
    if date_idx not in persistence_cache:
        print('Not precomputed for this index')
        return
    gudhi.plot_persistence_barcode(persistence_cache[date_idx])
    plt.title(f"Barcode — {str(dates[date_idx])[:10]}")
    plt.show()

widgets.interact(persistence_widget, date_idx=(0, len(dates)-1, 1))

## Note on Orthogonality

The **rolling complexes** (Section 7) and **persistent homology** (Section 10) are orthogonal constructions:
- Rolling complexes: fixed radius, varying time → one complex per date
- Persistence: fixed time, varying radius → a filtration at one date

Together they give a 2D picture: how the topology at each *scale* (radius) evolves over *time*.